# Exam Project - Introduction to Social Data Science
August 28, 2024

## Project: Forecasting Vote Counts for Danish Borgerforslag

## Group 7:
- Oliver Nyrop Weeks (vsn684)
- Sofus Galavits Møller (qvc730)
- Victor V. Kristensen (gcp458)
- Jonas T. Schmidt (mcp656)

## Load modules

In [26]:
# Module Imports
import selenium                # For navigating borgerforslag.dk
import requests                # For web scraping
import pandas as pd            # For data manipulation and analysis
from bs4 import BeautifulSoup  # For parsing HTML content
import time                    # For managing time delays during scraping
import tqdm                    # For displaying progress bars during scraping
import random                  # For randomizing delays in scraping to avoid detection
import pprint                  # For neatly displaying JSON code
import re                      # For pattern recognition in extracted HTML
from pathlib import Path       # For handling file paths
import csv                     # For exporting data to .csv files
import json                    # For exporting data to .json files
import os                      # For interacting with the operating system (e.g., file paths, environment variables)

# Class and Function Imports
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By                # For using CSS selectors (e.g., cookie-clicking)
from selenium.webdriver.support.ui import WebDriverWait    # For implementing explicit waits
from selenium.webdriver.support import expected_conditions as EC 
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.keys import Keys            # For simulating keyboard actions (e.g., RETURN key)
from selenium.common.exceptions import NoSuchElementException


## Connecting to Borgerforslag.dk

We first need to connect to Borgerforslag.dk to gather our data. This is accomplished using Selenium. We use the Selenium Chrome Driver to navigate the website, automating tasks such as selecting the correct site path and accepting cookies. Additionally, we adjust the settings to display all proposals, including the expired ones.

In [4]:
# Set Chrome options to disable the search engine choice screen
chrome_options = Options()
chrome_options.add_argument("--disable-search-engine-choice-screen")

# Initialize the Selenium Chrome driver with the specified options
driver = webdriver.Chrome(options=chrome_options)

# URL for Borgerforslag.dk
url_Borgerforslag = "https://borgerforslag.dk/"

# Use the .get() method to open the URL in the Selenium Chrome browser
driver.get(url_Borgerforslag)

# Attempt to locate and click the cookie consent button
try:
    cookie = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, 'CybotCookiebotDialogBodyLevelButtonLevelOptinAllowallSelection'))
    )
    cookie.click()
except TimeoutException:
    print("Element not found within the specified wait time.")

# Attempt to select "Alle" instead of "Igangværende" proposals
try:
    # Locate and click the filter dropdown
    click_1 = WebDriverWait(driver, 5).until(
        EC.presence_of_element_located((By.ID, 'react-select-filter1--value-item'))
    )
    click_1.click()

    # Locate and select the "Alle" option
    click_2 = WebDriverWait(driver, 3).until(
        EC.presence_of_element_located((By.ID, 'react-select-filter1--option-0'))
    )
    click_2.click()
except TimeoutException:
    print("Element not found within the specified wait time.")


We have now connected to the site, accepted cookies, and selected all proposals instead of just the active ones. We will now scroll (and click) through the site to be able to view all the elements at once. We create a loop that clicks through this, and stops when no more borgerforslag can be loaded (when the "load more" button disappears).

In [5]:
# Initialize a flag to control the loop
alarm = False  # Defining a stop_alarm

# Loop to navigate through additional pages
while not alarm:
    try:
        # Attempt to locate and click the "load more" button
        click_3 = driver.find_element(By.CSS_SELECTOR, 'button[class="dFsu8t fYY1lZ vFact_DoNotReadAloud _3-IJkM _3CrCss"]')
        click_3.click()

        # Introduce a random delay between clicks to mimic human interaction
        time.sleep(random.uniform(1, 5))
    except NoSuchElementException:
        # If the button is not found, stop the loop
        alarm = True
        print("No more pages to go through")


No more pages to go through


We have now navigated to the bottom of the page in our Selenium Chrome browser, successfully loading all of the borgerforslag. This means we can begin the scraping process, which will collect all of the HTML, including the individual links to each borgerforslag.

In [21]:
from bs4 import BeautifulSoup

# Parse the page source with BeautifulSoup using 'lxml' parser
soup = BeautifulSoup(driver.page_source, 'lxml')

# Find all <a> tags with the specified class
all_sites = soup.find_all('a', class_='lQq327')

# Extract the href attributes from each <a> tag and store them in a list
links = [site['href'] for site in all_sites]

# Print the number of borgerforslag found
print(f"Number of borgerforslag found: {len(links)}\n")

# Print the list of links
for link in links:
    print(link)


Number of borgerforslag found: 1847

https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18153
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18099
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18075
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18046
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18044
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18028
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18016
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-17997
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18050
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18032
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18062
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18029
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18030
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-17960
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-17930
https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-17914
https://borgerforslag.dk/se-og-stoe

Now that we have all the links to the borgerforslag, we can start scraping the necessary information from each proposal.

In the following block, we define a logging function to record key details about our scraping process for documentation purposes.

In [27]:
# Define the log function to gather and record log information
def log(response, logfile, url, output_path=os.getcwd()):
    # Open or create the log file
    if os.path.isfile(logfile):  # If the log file exists, open it for appending
        log = open(logfile, 'a')
    else:  # If the log file does not exist, create it with headers
        log = open(logfile, 'w')
        header = ['timestamp', 'status_code', 'length', 'url', 'output_file']
        log.write(';'.join(header) + "\n")  # Write headers and move to the next line
        
    # Gather log information
    status_code = response.status_code  # Status code from the HTTP response
    timestamp = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time()))  # Current local time
    length = len(response.text)  # Length of the HTML content
    
    # Append the gathered information to the log file
    with open(logfile, 'a') as log:
        log.write(f'{timestamp};{status_code};{length};{url};{output_path}' + "\n")  # Log the details and move to a new line

With the logging function defined, we can now proceed to scrape data from the individual borgerforslag links.

In [67]:
import requests
import time
import tqdm
import os

# Directory to save the log file and output file
output_dir = 'borgerforslag_batches'
os.makedirs(output_dir, exist_ok=True)

# Log file name
log_filename = os.path.join('logfile_borgerforslag.csv')
output_filename = os.path.join(output_dir, 'html_content.txt')

# List to store the HTML content from each URL
list_htmls = []

# Limit the range to the first 5 links for testing
test_links = links[:5]

# Loop through each URL in the links list
for i in tqdm.tqdm(test_links):
    try:
        # Send an HTTP GET request to the current URL with custom headers
        response = requests.get(i, headers={"Name": "Oliver Nyrop Weeks", "Email": "vsn684@alumni.ku.dk"})
        
        # Get the HTML content from the response
        html = response.text
        
        # Append the HTML content to the list
        list_htmls.append(html)
        
        # Log the request details
        log(response, log_filename, i)
        
        # Sleep for 0.5 seconds to avoid overloading the server
        time.sleep(random.uniform(1, 2))
    except Exception as e:
        # If an error occurs, print the URL and the error
        print(f"Error with URL: {i}")
        print(e)
        # Log the error with a status code of 0 (indicating failure)
        log(response, log_filename, i)

# Save the list of HTML content to a text file
with open(output_filename, 'w', encoding='utf-8') as output_file:
    for html_content in list_htmls:
        # Write each HTML content on a new line
        output_file.write(html_content + "\n")


100%|██████████| 5/5 [00:09<00:00,  1.80s/it]


In [68]:
list_htmls

['\n\n<!DOCTYPE html>\n<html lang="da-DK">\n<head>\n    <title>Afskaf inklusionsloven</title>\n\n\n    <meta charset="utf-8" />\n\n    \n<script id="Cookiebot" data-cbid="51f634e9-7d87-4212-af57-edd0e26f6f06" data-blockingmode="none" type="text/javascript" src="https://consent.cookiebot.com/uc.js"></script>\n\n<script type="text/javascript">\n       window.__THIRD_PARTY_KEYS = { sentry: "https://984e0a1cc92a49acaf5b315f1f3f1cd1@sentry.io/216509" };\n\n       window.addEventListener(\'CookiebotOnAccept\', function (e) {\n           if (Cookiebot.consent.statistics) {\n               // load app insight after consent\n               var appInsights = window.appInsights || function (a) {\n                   function b(a) { c[a] = function () { var b = arguments; c.queue.push(function () { c[a].apply(c, b) }) } } var c = { config: a }, d = document, e = window; setTimeout(function () { var b = d.createElement("script"); b.src = a.url || "https://az416426.vo.msecnd.net/scripts/a/ai.0.js", d

In [ ]:
# Directory to save the output files
output_dir = 'borgerforslag_batches'
os.makedirs(output_dir, exist_ok=True)

# Set the number of links to process in each batch
batch_size = 5

# Start index (set this manually or pass it as a parameter)
start_index = 0  # Set to 0 to start from the beginning, or another number to start elsewhere

# Log file name
log_filename = 'logfile_borgerforslag.csv'

# This list will store the HTML content across all batches
all_storing_raw_html = []

# Loop through the URLs starting from the specified index
for i, url in enumerate(tqdm.tqdm(links[start_index:]), start=start_index + 1):
    try:
        # Send an HTTP GET request to the current borgerforslag URL
        html_borger = requests.get(url, headers={'Navn':'Sofus Møller', 'Email': 'qvc730@samf.ku.dk'})
    except Exception as e:
        # If an error occurs, print the URL and the error
        print(url)
        print(e)
        continue  # Skip to the next URL
    
    # Append the raw HTML content to both the current batch and the overall list
    storing_raw_html.append(html_borger.text)
    all_storing_raw_html.append(html_borger.text)
    
    # Log the request details using the log function
    log(html_borger, os.path.join(output_dir, log_filename), url)
    
    # Sleep for a random time between 1 and 3 seconds to avoid overloading the server
    time.sleep(random.uniform(1, 3))
    
    # Save and reset after every batch_size links
    if i % batch_size == 0:
        batch_number = i // batch_size
        with open(os.path.join(output_dir, f'storing_raw_html_batch_{batch_number}.json'), 'w', encoding='utf-8') as l:
            json.dump(storing_raw_html, l)
        storing_raw_html = []  # Clear the list for the next batch

# Save any remaining data that didn't fill the last batch
if storing_raw_html:
    batch_number = (i // batch_size) + 1
    with open(os.path.join(output_dir, f'storing_raw_html_batch_{batch_number}.json'), 'w', encoding='utf-8') as l:
        json.dump(storing_raw_html, l)

# Join the full list into a single string, with each HTML block separated by nothing
combined_text = "".join(all_storing_raw_html)

# Write the combined text to a file
with open('output.txt', 'w', encoding='utf-8') as file:
    file.write(combined_text)


In [59]:
import json
from bs4 import BeautifulSoup

# Load the JSON file
with open('borgerforslag_batches/storing_raw_html_batch_1.json', 'r') as file:
    data = json.load(file)

# Assuming the JSON structure contains a list with each item containing the raw HTML
for item in data:
    html_content = item['html']  # Adjust the key if necessary

    # Parse the HTML
    soup = BeautifulSoup(html_content, 'html.parser')

    # Extract the title
    title = soup.title.string if soup.title else "No title found"

    # Extract the start date
    start_date_tag = soup.find('div', text='Startdato')
    start_date = start_date_tag.find_next('strong').text if start_date_tag else "No start date found"

    # Extract the end date
    end_date_tag = soup.find('div', text='Slutdato')
    end_date = end_date_tag.find_next('strong').text if end_date_tag else "No end date found"

    # Extract the main text (the proposal content)
    main_text_tag = soup.find('div', class_='cc552X')
    main_text = main_text_tag.text.strip() if main_text_tag else "No main text found"

    # Extract the number of supporters
    supporters_tag = soup.find('div', text='Antal støtter')
    supporters = supporters_tag.find_next('strong').text if supporters_tag else "No supporter count found"

    # Extract the proposal ID
    proposal_id_tag = soup.find('span', class_='_3jrW-w')
    proposal_id = proposal_id_tag.text.replace('ID: ', '') if proposal_id_tag else "No proposal ID found"

    # Output the extracted information
    print(f"Title: {title}")
    print(f"Start Date: {start_date}")
    print(f"End Date: {end_date}")
    print(f"Main Text: {main_text}")
    print(f"Number of Supporters: {supporters}")
    print(f"Proposal ID: {proposal_id}")
    print("-" * 50)


TypeError: string indices must be integers, not 'str'

In [60]:
# defining the file path to raw_html
raw_data = Path.cwd() / "raw_data" / "raw_html.csv"

# making into df
df_storing_raw_html = pd.DataFrame(storing_raw_html)

# Ensure the directory exists
raw_data.parent.mkdir(parents=True, exist_ok=True)

# Check if the file already exists
if not raw_data.exists():
    # If the file doesn't exist, save the DataFrame to CSV
    df_storing_raw_html.to_csv(raw_data, index=False)
    print(f"Data written to '{raw_data}' successfully.")
else:
    print(f"File '{raw_data}' already exists. Operation aborted.")

AttributeError: 'NoneType' object has no attribute 'string'